# Croatian binary detector: threshold tuning

The Croatian XLM-R binary detector trained on translated TherapistQA achieved 0.74 weighted F1 in-distribution, but the 30-example deployment validation caught only 3/10 obviously-distorted Croatian thoughts. Inspection of the confidence distribution showed the model's probability mass is shifted toward No Distortion: 21/30 examples landed at prob(D) < 0.3.

This shift suggests the model is making correct *relative* judgements (Distorted is ranked higher than ND on most pairs) but its decision threshold of 0.5 is too high for the Croatian operating regime. This notebook tunes the threshold on the Croatian validation set by sweeping it from 0.10 to 0.50 in 0.05 steps and selecting the value that maximises validation F1_weighted. The chosen threshold is then re-applied to the 30-example deployment test from `tqa_pr_pipeline_arch_deploy_test_hr.ipynb` to measure the effect.


In [1]:
# === Setup ===
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.nn.functional import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    f1_score, accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

# === Paths — adjust if your saved model lives elsewhere ===
MODELS = Path("../../models")
BINARY_CHECKPOINT = MODELS / "05_01_cro_therapistqa_binary_xlm" / "model"

DATA_DIR = Path("../../processed")    # adjust to the directory containing the bilingual CSVs
VAL_CSV  = DATA_DIR / "therapistqa_bilingual_val.csv"
TEST_CSV = DATA_DIR / "therapistqa_bilingual_test.csv"

RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 512

print(f"Device: {device}")
print(f"Binary checkpoint: {BINARY_CHECKPOINT}")
print(f"Val CSV:  {VAL_CSV}")
print(f"Test CSV: {TEST_CSV}")
for p in (BINARY_CHECKPOINT, VAL_CSV, TEST_CSV):
    assert p.exists(), f"Missing: {p}"

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Device: cpu
Binary checkpoint: ../../models/05_01_cro_therapistqa_binary_xlm/model
Val CSV:  ../../processed/therapistqa_bilingual_val.csv
Test CSV: ../../processed/therapistqa_bilingual_test.csv


In [2]:
# === Load the Croatian binary detector ===
tokenizer = AutoTokenizer.from_pretrained(BINARY_CHECKPOINT)
model     = AutoModelForSequenceClassification.from_pretrained(BINARY_CHECKPOINT)
model.to(device).eval()

id2label = {i: model.config.id2label[i] for i in sorted(model.config.id2label, key=int)}
print(f"Labels: {id2label}")
assert id2label == {0: "No Distortion", 1: "Distorted"}, f"Unexpected labels: {id2label}"

Labels: {0: 'No Distortion', 1: 'Distorted'}


In [3]:
# === Get prob(distorted) on the entire Croatian val set and test set ===
def predict_probs(df, text_col="patient_question_hr", batch_size=32):
    """Return numpy array of prob(distorted) for every row in df."""
    df = df.dropna(subset=[text_col]).reset_index(drop=True)
    probs = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch = df[text_col].iloc[i:i+batch_size].astype(str).tolist()
            enc = tokenizer(batch, truncation=True, max_length=MAX_LEN,
                            padding=True, return_tensors="pt").to(device)
            logits = model(**enc).logits
            p = softmax(logits, dim=-1).cpu().numpy()
            probs.extend(p[:, 1].tolist())
    return df, np.array(probs)

# Validation set
df_val_raw = pd.read_csv(VAL_CSV)
df_val, val_probs = predict_probs(df_val_raw)
val_labels = (df_val["dominant_distortion"] != "No Distortion").astype(int).values

# Test set
df_test_raw = pd.read_csv(TEST_CSV)
df_test, test_probs = predict_probs(df_test_raw)
test_labels = (df_test["dominant_distortion"] != "No Distortion").astype(int).values

print(f"Validation: {len(df_val)} rows, mean prob(D) = {val_probs.mean():.3f}")
print(f"Test:       {len(df_test)} rows, mean prob(D) = {test_probs.mean():.3f}")
print(f"\nDistribution of val prob(D):")
for lo, hi in [(0.0, 0.2), (0.2, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 0.8), (0.8, 1.0)]:
    n_total = ((val_probs >= lo) & (val_probs < hi)).sum()
    n_pos   = ((val_probs >= lo) & (val_probs < hi) & (val_labels == 1)).sum()
    print(f"  [{lo:.1f}, {hi:.1f}): {n_total:>3}  of which Distorted = {n_pos:>3}  ({100*n_pos/max(n_total,1):.0f}%)")

Validation: 379 rows, mean prob(D) = 0.650
Test:       379 rows, mean prob(D) = 0.683

Distribution of val prob(D):
  [0.0, 0.2): 105  of which Distorted =  34  (32%)
  [0.2, 0.4):  20  of which Distorted =  10  (50%)
  [0.4, 0.5):   5  of which Distorted =   0  (0%)
  [0.5, 0.6):   6  of which Distorted =   5  (83%)
  [0.6, 0.8):  17  of which Distorted =  13  (76%)
  [0.8, 1.0): 226  of which Distorted = 177  (78%)


In [4]:
# === Sweep thresholds on the validation set ===
# For each candidate threshold, compute:
#   - precision / recall / F1 of the Distorted class
#   - precision / recall / F1 of the No Distortion class
#   - weighted F1, macro F1, accuracy
# Pick the threshold that maximises F1_weighted on validation
# (matches the metric_for_best_model used during training).

THRESHOLDS = np.arange(0.10, 0.55, 0.05)
rows = []
for thr in THRESHOLDS:
    preds = (val_probs >= thr).astype(int)
    p_d, r_d, f_d, _ = precision_recall_fscore_support(
        val_labels, preds, average="binary", pos_label=1, zero_division=0
    )
    p_nd, r_nd, f_nd, _ = precision_recall_fscore_support(
        val_labels, preds, average="binary", pos_label=0, zero_division=0
    )
    rows.append({
        "threshold":   round(float(thr), 2),
        "acc":         accuracy_score(val_labels, preds),
        "f1_weighted": f1_score(val_labels, preds, average="weighted", zero_division=0),
        "f1_macro":    f1_score(val_labels, preds, average="macro",    zero_division=0),
        "prec_D":      p_d,
        "rec_D":       r_d,
        "f1_D":        f_d,
        "prec_ND":     p_nd,
        "rec_ND":      r_nd,
        "f1_ND":       f_nd,
        "n_called_D":  int(preds.sum()),
        "n_called_ND": int((preds == 0).sum()),
    })

sweep = pd.DataFrame(rows)
best_row = sweep.loc[sweep["f1_weighted"].idxmax()]
best_threshold = float(best_row["threshold"])

print("Threshold sweep on validation set:")
print(sweep.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\nBest threshold (max F1_weighted): {best_threshold}")
print(f"  acc={best_row['acc']:.3f}  f1_weighted={best_row['f1_weighted']:.3f}  "
      f"f1_macro={best_row['f1_macro']:.3f}")
print(f"  rec_D={best_row['rec_D']:.3f}  rec_ND={best_row['rec_ND']:.3f}")

sweep.to_csv(RESULTS_DIR / "threshold_sweep_val_hr.csv", index=False)

Threshold sweep on validation set:
 threshold   acc  f1_weighted  f1_macro  prec_D  rec_D  f1_D  prec_ND  rec_ND  f1_ND  n_called_D  n_called_ND
     0.100 0.731        0.712     0.678   0.734  0.900 0.808    0.721   0.443  0.549         293           86
     0.150 0.736        0.724     0.694   0.747  0.879 0.808    0.704   0.493  0.580         281           98
     0.200 0.728        0.718     0.689   0.748  0.858 0.799    0.676   0.507  0.580         274          105
     0.250 0.731        0.724     0.698   0.758  0.841 0.798    0.667   0.543  0.598         265          114
     0.300 0.728        0.722     0.697   0.760  0.833 0.794    0.658   0.550  0.599         262          117
     0.350 0.728        0.724     0.699   0.764  0.824 0.793    0.653   0.564  0.605         258          121
     0.400 0.728        0.725     0.701   0.768  0.816 0.791    0.648   0.579  0.611         254          125
     0.450 0.734        0.731     0.708   0.774  0.816 0.794    0.654   0.593  0.622 

In [5]:
# === Apply the chosen threshold to the test set, compare to threshold=0.5 ===
preds_05    = (test_probs >= 0.5).astype(int)
preds_best  = (test_probs >= best_threshold).astype(int)

def report(name, y, p):
    acc  = accuracy_score(y, p)
    f1_w = f1_score(y, p, average="weighted", zero_division=0)
    f1_m = f1_score(y, p, average="macro",    zero_division=0)
    p_d, r_d, f_d, _ = precision_recall_fscore_support(y, p, average="binary",
                                                       pos_label=1, zero_division=0)
    p_nd, r_nd, f_nd, _ = precision_recall_fscore_support(y, p, average="binary",
                                                          pos_label=0, zero_division=0)
    print(f"\n{name}")
    print(f"  Accuracy:        {acc:.4f}")
    print(f"  F1 weighted:     {f1_w:.4f}")
    print(f"  F1 macro:        {f1_m:.4f}")
    print(f"  Distorted:       prec={p_d:.3f}  rec={r_d:.3f}  f1={f_d:.3f}")
    print(f"  No Distortion:   prec={p_nd:.3f}  rec={r_nd:.3f}  f1={f_nd:.3f}")
    print(f"  Confusion matrix [[TN, FP], [FN, TP]]:")
    cm = confusion_matrix(y, p)
    for row in cm:
        print(f"    {row}")

report(f"Test set @ threshold = 0.50 (default)",     test_labels, preds_05)
report(f"Test set @ threshold = {best_threshold:.2f} (tuned)", test_labels, preds_best)


Test set @ threshold = 0.50 (default)
  Accuracy:        0.7388
  F1 weighted:     0.7337
  F1 macro:        0.7097
  Distorted:       prec=0.769  rec=0.837  f1=0.802
  No Distortion:   prec=0.672  rec=0.571  f1=0.618
  Confusion matrix [[TN, FP], [FN, TP]]:
    [80 60]
    [ 39 200]

Test set @ threshold = 0.50 (tuned)
  Accuracy:        0.7388
  F1 weighted:     0.7337
  F1 macro:        0.7097
  Distorted:       prec=0.769  rec=0.837  f1=0.802
  No Distortion:   prec=0.672  rec=0.571  f1=0.618
  Confusion matrix [[TN, FP], [FN, TP]]:
    [80 60]
    [ 39 200]


## Re-run the deployment evaluation with the tuned threshold

Same 30 hand-curated Croatian examples as `tqa_pr_pipeline_arch_deploy_test_hr.ipynb`. Pasting them here so this notebook is self-contained.

This is the headline number: does the tuned threshold catch more obviously-distorted thoughts at deployment? And does it false-fire on the neutral ones?

In [6]:
EXAMPLES_HR = [
    # Obviously distorted (one per category)
    ("distorted", "Ako ne dobijem ovo promaknuće, cijela mi je karijera promašaj.",
     "All-or-nothing thinking"),
    ("distorted", "Pogriješio sam jednu stvar u prezentaciji i sada će svi misliti da sam nesposoban.",
     "Catastrophizing"),
    ("distorted", "Rekla je da joj se sviđa moj rad, ali sigurno je samo bila pristojna.",
     "Discounting the positive"),
    ("distorted", "Jednostavno znam da će ovaj spoj proći loše.",
     "Fortune-telling"),
    ("distorted", "Nije mi odgovorio na poruku, sigurno je ljut na mene.",
     "Mind reading"),
    ("distorted", "Zaboravio sam poslati email. Takav sam idiot.",
     "Labeling"),
    ("distorted", "Svi su pljeskali osim jedne osobe koja je izgledala dosadno. Cijelo izlaganje je bila katastrofa.",
     "Mental filtering"),
    ("distorted", "Upropastio sam ovaj razgovor za posao. Nikada neću naći zaposlenje.",
     "Overgeneralization"),
    ("distorted", "Partner mi je loše raspoložen. Sigurno sam ja nešto napravio.",
     "Personalization"),
    ("distorted", "Trebao bih biti puno dalje u karijeri u ovim godinama.",
     "Should statements"),

    # Non-distorted
    ("nondistorted", "Trebao bih nazvati mamu ovaj vikend.", "No Distortion"),
    ("nondistorted", "Izgleda da bi moglo padati kasnije.", "No Distortion"),
    ("nondistorted", "Sutra u 15 sati imam sastanak.", "No Distortion"),
    ("nondistorted", "Kava je danas dobra.", "No Distortion"),
    ("nondistorted", "Želim ove godine više čitati.", "No Distortion"),
    ("nondistorted", "Bole me leđa nakon cijelog dana sjedenja.", "No Distortion"),
    ("nondistorted", "Umoran sam i mislim da ću ranije u krevet.", "No Distortion"),
    ("nondistorted", "Rok je u petak pa moram završiti izvještaj do četvrtka.", "No Distortion"),
    ("nondistorted", "Nisam se složio s onim što je rekla, ali razumijem njezin stav.", "No Distortion"),
    ("nondistorted", "Danas sam nešto napredovao, iako još ima posla.", "No Distortion"),

    # Ambiguous
    ("ambiguous", "Nije me pozdravio na sastanku.", "uncertain"),
    ("ambiguous", "Brinem se da neću imati dovoljno novca za mirovinu.", "uncertain"),
    ("ambiguous", "Ako sredstva ne stignu, projekt je gotov.", "uncertain"),
    ("ambiguous", "Šef me ovog tjedna izbjegavao pogledom.", "uncertain"),
    ("ambiguous", "Nisam smio izgubiti živce s njim.", "uncertain"),
    ("ambiguous", "S obzirom na trenutno tržište, posao u mojoj struci bit će teško naći.", "uncertain"),
    ("ambiguous", "Općenito mi je dobro išlo, ali stalno razmišljam o onom jednom lošem odgovoru.", "uncertain"),
    ("ambiguous", "Mislim da nisam dovoljno kvalificiran za ovu poziciju.", "uncertain"),
    ("ambiguous", "Tim nije stigao do roka, a ja sam bila voditeljica.", "uncertain"),
    ("ambiguous", "Nešto ne valja, ali ne mogu točno odrediti što.", "uncertain"),
]

# Predict prob(D) for each example
deployment_probs = []
for intent, text, _ in EXAMPLES_HR:
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    p = softmax(logits, dim=-1).cpu().numpy()[0]
    deployment_probs.append(float(p[1]))
deployment_probs = np.array(deployment_probs)

print("Deployment prob(D) distribution:")
print(pd.Series(deployment_probs).describe().round(3))

Deployment prob(D) distribution:
count    30.000
mean      0.330
std       0.363
min       0.037
25%       0.057
50%       0.163
75%       0.488
max       0.979
dtype: float64


In [7]:
# Compare deployment behaviour at threshold=0.5 vs threshold=best
def deployment_summary(thr, probs, examples):
    calls = (probs >= thr).astype(int)
    distorted_idx    = [i for i, e in enumerate(examples) if e[0] == "distorted"]
    nondistorted_idx = [i for i, e in enumerate(examples) if e[0] == "nondistorted"]
    ambiguous_idx    = [i for i, e in enumerate(examples) if e[0] == "ambiguous"]
    return {
        "threshold":               thr,
        "distorted_caught":        int(sum(calls[i] for i in distorted_idx)),
        "distorted_total":         len(distorted_idx),
        "nondistorted_correct":    int(sum(1 - calls[i] for i in nondistorted_idx)),
        "nondistorted_total":      len(nondistorted_idx),
        "ambiguous_called_D":      int(sum(calls[i] for i in ambiguous_idx)),
        "ambiguous_total":         len(ambiguous_idx),
    }

summary_05   = deployment_summary(0.50,           deployment_probs, EXAMPLES_HR)
summary_best = deployment_summary(best_threshold, deployment_probs, EXAMPLES_HR)

print(f"{'Metric':<40} {'thr=0.50':>10} {'thr='+str(round(best_threshold,2)):>10}")
print("-" * 64)
print(f"{'Distorted caught (of 10)':<40} "
      f"{summary_05['distorted_caught']:>10} {summary_best['distorted_caught']:>10}")
print(f"{'Non-distorted correct (of 10)':<40} "
      f"{summary_05['nondistorted_correct']:>10} {summary_best['nondistorted_correct']:>10}")
print(f"{'Ambiguous flagged as Distorted (of 10)':<40} "
      f"{summary_05['ambiguous_called_D']:>10} {summary_best['ambiguous_called_D']:>10}")

# Per-example detail at the new threshold — which distortions now fire and which don't
print(f"\nPer-example detail at threshold = {best_threshold:.2f}:")
print(f"\n  Distorted (target = Distorted):")
for i, (intent, text, expected) in enumerate(EXAMPLES_HR):
    if intent != "distorted":
        continue
    mark = "✓" if deployment_probs[i] >= best_threshold else "✗"
    print(f"    {mark} prob(D)={deployment_probs[i]:.2f}  [{expected}] {text[:60]}")

print(f"\n  Non-distorted (target = ND):")
for i, (intent, text, expected) in enumerate(EXAMPLES_HR):
    if intent != "nondistorted":
        continue
    mark = "✓" if deployment_probs[i] < best_threshold else "✗"
    print(f"    {mark} prob(D)={deployment_probs[i]:.2f}  {text[:60]}")

Metric                                     thr=0.50    thr=0.5
----------------------------------------------------------------
Distorted caught (of 10)                          4          4
Non-distorted correct (of 10)                    10         10
Ambiguous flagged as Distorted (of 10)            4          4

Per-example detail at threshold = 0.50:

  Distorted (target = Distorted):
    ✓ prob(D)=0.97  [All-or-nothing thinking] Ako ne dobijem ovo promaknuće, cijela mi je karijera promaša
    ✓ prob(D)=0.97  [Catastrophizing] Pogriješio sam jednu stvar u prezentaciji i sada će svi misl
    ✗ prob(D)=0.04  [Discounting the positive] Rekla je da joj se sviđa moj rad, ali sigurno je samo bila p
    ✓ prob(D)=0.93  [Fortune-telling] Jednostavno znam da će ovaj spoj proći loše.
    ✗ prob(D)=0.04  [Mind reading] Nije mi odgovorio na poruku, sigurno je ljut na mene.
    ✗ prob(D)=0.14  [Labeling] Zaboravio sam poslati email. Takav sam idiot.
    ✗ prob(D)=0.34  [Mental filtering] Svi s

In [8]:
# Save all results to disk
deployment_df = pd.DataFrame([
    {
        "intent": intent,
        "text": text,
        "expected": expected,
        "prob_distorted": float(deployment_probs[i]),
        "call_thr_0.5":  "Distorted" if deployment_probs[i] >= 0.5 else "No Distortion",
        "call_thr_best": "Distorted" if deployment_probs[i] >= best_threshold else "No Distortion",
    }
    for i, (intent, text, expected) in enumerate(EXAMPLES_HR)
])
deployment_df.to_csv(RESULTS_DIR / "deployment_with_tuned_threshold_hr.csv", index=False)

summary_df = pd.DataFrame([summary_05, summary_best])
summary_df.to_csv(RESULTS_DIR / "deployment_threshold_comparison_hr.csv", index=False)

print(f"Saved deployment per-example data to deployment_with_tuned_threshold_hr.csv")
print(f"Saved summary to deployment_threshold_comparison_hr.csv")

Saved deployment per-example data to deployment_with_tuned_threshold_hr.csv
Saved summary to deployment_threshold_comparison_hr.csv
